# Do time and space improve the prediction of future returns?

Final project of the Stock Markets Analytics course, built as release 1.0.0 of `tsip-core`. This notebook runs the whole flow by calling the project's modules in `src/tsip/capstone/`; it contains no logic of its own. Every figure below is printed by that code (the written reading is in `docs/CAPSTONE.md`).

**Question.** TSIP describes where price stands in time (calendar coordinates) and in space (against the highs and lows of prior days, weeks, months and quarters). Do those features improve the out-of-sample prediction of the direction of the next 5 sessions for SPY, QQQ, GLD, Bitcoin and Ethereum, beyond the course's standard features, and does a strategy built on the predictions beat buy and hold after costs?

**Protocol.** Specification `specs/0028-time-space-features-for-return-prediction.md`; hypothesis H-2026-016 registered in `registry/hypotheses.json` before any model was trained. Training up to 2021, validation 2022–2023 (all choices), test from 2024 (read once, by a logged command).

**Modes.** `MODE = "file"` reads the stored copy (no network); `MODE = "download"` fetches fresh data from yfinance and FRED first. Data are stored outside Git in `~/tsip_capstone_data` (or `TSIP_CAPSTONE_DATA`).

In [1]:
MODE = "file"  # or "download"

import contextlib
import io
import json
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from tsip.capstone import analysis, data, evaluate, features, paper_order, run_models, simulate

store = data.default_store()
results = store / "results"
pd.set_option("display.width", 140)
# The earliest tuning folds (before 2014) have reference columns without values (Bitcoin did not trade);
# the imputer skips them and warns. The warning is expected and silenced here.
warnings.filterwarnings("ignore", message="Skipping features without any observed values")


def quiet(fn, *args):
    """Run a module's command-line entry point without its progress lines."""
    with contextlib.redirect_stdout(io.StringIO()):
        fn(*args)

## 1. Data
One daily frame for the five instruments with reference markets (S&P 500, Nasdaq-100, DAX, gold, oil, Bitcoin, VIX) and FRED macro series, each joined only from values public at the row's close.

In [2]:
frame, summary = data.load(MODE, store)
{k: v for k, v in summary.items() if k != "store"}

{'created_utc': '2026-11-10T00:37:33+00:00',
 'source_rows': {'prices': 71800, 'fred': 36226},
 'unified_rows': 26759,
 'unified_columns': 20,
 'first_date': '2000-01-03',
 'last_date': '2026-11-09',
 'rows_by_ticker': {'BTC-USD': 4436,
  'ETH-USD': 3287,
  'GLD': 5528,
  'QQQ': 6754,
  'SPY': 6754}}

## 2. Features
122 columns in named sets. No feature on a row changes when later rows are deleted (`tests/test_capstone_features.py`).

In [3]:
feats, sets = features.build_features(frame)
pd.Series({name: len(cols) for name, cols in sets.items()}, name="columns")

TO_PREDICT     4
LECTURE       60
TIME          11
SPACE         40
DUMMIES        5
Name: columns, dtype: int64

### Exploratory view (whole history, descriptive only)
Share of positive 5-session periods by instrument, and by weekday (a time feature) and by position of the close in the prior week's range (a space feature).

In [4]:
known = feats[feats["is_positive_growth_5d"].notna()].copy()
known["weekday"] = known["weekday"].map(dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])))
known["prior_week_position"] = pd.cut(known["space_week_position"], [-10, 0, 0.25, 0.5, 0.75, 1, 10],
                                      labels=["below low", "0-25%", "25-50%", "50-75%", "75-100%", "above high"])
display((known.groupby("ticker")["is_positive_growth_5d"].mean() * 100).round(1).rename("positive 5-session %"))
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
display((known.pivot_table(index="weekday", columns="ticker", values="is_positive_growth_5d") * 100).round(1).reindex(days))
display((known.pivot_table(index="prior_week_position", columns="ticker", values="is_positive_growth_5d", observed=True) * 100).round(1))

ticker
BTC-USD    54.1
ETH-USD    51.8
GLD        55.2
QQQ        57.2
SPY        58.0
Name: positive 5-session %, dtype: float64

ticker,BTC-USD,ETH-USD,GLD,QQQ,SPY
weekday,,,,,
Mon,53.9,50.1,54.0,57.2,57.8
Tue,53.4,51.4,55.7,58.1,59.1
Wed,52.9,50.2,54.3,58.8,59.1
Thu,55.0,54.2,55.8,56.1,58.0
Fri,55.1,55.7,55.9,55.8,55.9
Sat,54.0,50.1,NaN,NaN,NaN
Sun,54.3,50.7,NaN,NaN,NaN


ticker,BTC-USD,ETH-USD,GLD,QQQ,SPY
prior_week_position,,,,,
below low,58.7,55.2,57.0,59.3,61.2
0-25%,50.5,50.0,51.4,56.3,56.1
25-50%,52.3,49.2,55.3,58.0,56.1
50-75%,53.1,48.7,57.1,56.6,60.1
75-100%,54.6,52.0,52.3,56.0,57.8
above high,54.8,53.7,55.8,57.0,56.6


### Correlation with the target
Spearman correlation of each feature with the return of the next 5 and 20 sessions, on training and validation rows only (the test period is not used for exploration). The 15 strongest.

In [5]:
for horizon in (5, 20):
    display(Markdown(f"**Next {horizon} sessions**"))
    display(analysis.correlations(feats, sets, horizon).round(3))

**Next 5 sessions**

,feature,set,spearman,rows
0,macro_ten_year_yield,LECTURE,-0.063,22522
1,growth_btc_usd_30d,LECTURE,0.062,12521
2,macro_cpi_yoy,LECTURE,-0.052,21035
3,growth_wti_oil_7d,LECTURE,-0.052,22167
4,growth_btc_usd_90d,LECTURE,0.049,12278
5,growth_wti_oil_365d,LECTURE,-0.047,20709
6,growth_brent_oil_365d,LECTURE,-0.045,16553
7,session_of_month,TIME,0.044,22522
8,space_month_untested_below_age,SPACE,0.043,21120
9,growth_brent_oil_7d,LECTURE,-0.043,18010


**Next 20 sessions**

,feature,set,spearman,rows
0,growth_wti_oil_365d,LECTURE,-0.110,20709
1,growth_brent_oil_365d,LECTURE,-0.109,16553
2,macro_cpi_yoy,LECTURE,-0.104,21035
3,macro_tbill_3m_change_30d,LECTURE,-0.096,22399
4,macro_ten_year_yield,LECTURE,-0.094,22522
5,growth_btc_usd_30d,LECTURE,0.089,12521
6,growth_btc_usd_90d,LECTURE,0.083,12278
7,growth_btc_usd_7d,LECTURE,0.069,12618
8,macro_ten_year_yield_change_30d,LECTURE,-0.059,22399
9,ma_ratio_200,LECTURE,0.057,21527


## 3. Models on the validation period
Decision tree, random forest, logistic regression and XGBoost, each tuned on time-ordered folds of the training period and scored on validation, on four feature sets. Pieces whose results exist are reused; from scratch the random forest takes about 15 minutes on a laptop (set `FAMILIES = ["logistic"]` for a quick run). The family with the highest mean AUC is selected: logistic regression.

In [6]:
FAMILIES = list(run_models.FAMILIES)
for horizon in (5, 20):
    folder = results / f"h{horizon}"
    quiet(run_models.fit, feats, sets, horizon, FAMILIES, list(run_models.M.FEATURE_SETS), folder)
    table = pd.DataFrame(run_models.summarise(folder, horizon)["table"]).set_index("family")
    display(Markdown(f"**Direction of the next {horizon} sessions, validation AUC**"))
    display(table.round(3))

**Direction of the next 5 sessions, validation AUC**

,LECTURE,LECTURE+TIME,LECTURE+SPACE,LECTURE+TIME+SPACE,mean,permuted_time_space,lift,lift_low,lift_high
family,,,,,,,,,
logistic,0.555,0.561,0.548,0.553,0.554,0.537,-0.002,-0.039,0.036
decision_tree,0.480,0.490,0.499,0.490,0.490,0.483,0.010,-0.069,0.086
xgboost,0.538,0.551,0.530,0.548,0.542,0.504,0.009,-0.032,0.047
random_forest,0.477,0.515,0.495,0.512,0.500,0.487,0.035,-0.003,0.066


**Direction of the next 20 sessions, validation AUC**

,LECTURE,LECTURE+TIME,LECTURE+SPACE,LECTURE+TIME+SPACE,mean,permuted_time_space,lift,lift_low,lift_high
family,,,,,,,,,
logistic,0.545,0.540,0.542,0.533,0.540,0.531,-0.012,-0.045,0.023
decision_tree,0.535,0.528,0.535,0.528,0.531,0.528,-0.007,-0.048,0.039
xgboost,0.541,0.536,0.541,0.497,0.529,0.491,-0.044,-0.099,0.013
random_forest,0.476,0.451,0.493,0.471,0.473,0.472,-0.006,-0.032,0.017


## 4. Strategy simulations on the validation period
Four strategies on the predictions and pullback buying, with costs per side and idle cash at the 3-month bill rate. Parameters are chosen here and frozen in `registry/capstone_choices_h5.json`. Read the random-walk control in `docs/CAPSTONE.md`: on two years, random walks also reach high Sharpe ratios.

In [7]:
quiet(simulate.main, ["--horizon", "5", "--store", str(store)])
display(Markdown(simulate.markdown(json.loads((results / "simulation_h5.json").read_text()))))

**LECTURE** (period 2022-01-01 to 2023-12-26)

| Strategy | Chosen parameters | CAGR (95% interval) | CAGR minus five equal (95%) | Sharpe | Max drawdown | Rolling 1y min / median | Exposure | Entries | CAGR, Sharpe with one-session delay |
|---|---|---|---|---:|---:|---|---:|---:|---|
| sim1 threshold | quantile 0.900, threshold 0.560 | +10.5% (+3.5 to +19.5) | -25.4 to +43.2 | 1.10 | -3.7% | +3.1% / +5.9% | 0.10 | 86 | +7.7%, 0.77 |
| sim2 top k | quantile 0.900, threshold 0.560, k 5 | +10.5% (+3.5 to +19.5) | -25.4 to +43.2 | 1.10 | -3.7% | +3.1% / +5.9% | 0.10 | 86 | +7.7%, 0.77 |
| sim3 stop and filter | threshold 0.560 | +4.9% (+1.2 to +8.9) | -31.6 to +36.7 | 0.43 | -2.3% | +3.3% / +4.9% | 0.05 | 57 | +5.6%, 0.65 |
| sim4 crypto long-short | quantile 0.900, upper 0.537, lower 0.330 | +39.1% (+9.8 to +92.0) | -11.7 to +109.6 | 1.33 | -11.9% | -2.6% / +52.1% | 0.20 | 108 | +19.9%, 0.72 |
| Buy and hold, five equal | - | -3.2% (-31.9 to +37.2) | - | -0.16 | -39.8% | -37.0% / +11.9% | 1.00 | 5 | - |
| Buy and hold, crypto | - | -14.0% (-64.1 to +104.6) | -36.8 to +72.7 | -0.01 | -68.9% | -66.6% / +12.2% | 1.00 | 2 | - |
| SPY with dividends | - | +1.7% (-21.5 to +28.5) | -18.7 to +23.7 | 0.01 | -24.5% | -19.0% / +9.7% | 1.00 | 1 | - |
| SPY price only | - | +0.1% (-22.9 to +27.1) | -20.5 to +22.1 | -0.07 | -25.4% | -20.3% / +7.9% | 1.00 | 1 | - |

Rotation control of sim1 (100 rotations): median Sharpe -0.28 (95% -1.31 to 1.07); share at or above the actual 0.03. Vector view of sim1: 294 bets of 100, hit rate 55.1%, mean net +0.82 per bet.

**LECTURE+TIME+SPACE** (period 2022-01-01 to 2023-12-26)

| Strategy | Chosen parameters | CAGR (95% interval) | CAGR minus five equal (95%) | Sharpe | Max drawdown | Rolling 1y min / median | Exposure | Entries | CAGR, Sharpe with one-session delay |
|---|---|---|---|---:|---:|---|---:|---:|---|
| sim1 threshold | quantile 0.500, threshold 0.448 | +28.0% (-2.2 to +65.5) | +8.8 to +55.7 | 1.26 | -18.8% | +8.4% / +43.0% | 0.51 | 227 | +16.3%, 0.74 |
| sim2 top k | quantile 0.500, threshold 0.448, k 5 | +28.0% (-2.2 to +65.5) | +8.8 to +55.7 | 1.26 | -18.8% | +8.4% / +43.0% | 0.51 | 227 | +16.3%, 0.74 |
| sim3 stop and filter | threshold 0.448 | +27.5% (+3.1 to +61.0) | +3.2 to +53.2 | 1.79 | -14.7% | +7.4% / +37.1% | 0.31 | 185 | +18.0%, 1.24 |
| sim4 crypto long-short | quantile 0.700, upper 0.490, lower 0.382 | +70.1% (-6.5 to +214.3) | -6.8 to +220.3 | 1.35 | -39.8% | +43.5% / +185.8% | 0.59 | 191 | +68.2%, 1.34 |
| Buy and hold, five equal | - | -3.2% (-31.9 to +37.2) | - | -0.16 | -39.8% | -37.0% / +11.9% | 1.00 | 5 | - |
| Buy and hold, crypto | - | -14.0% (-64.1 to +104.6) | -36.8 to +72.7 | -0.01 | -68.9% | -66.6% / +12.2% | 1.00 | 2 | - |
| SPY with dividends | - | +1.7% (-21.5 to +28.5) | -18.7 to +23.7 | 0.01 | -24.5% | -19.0% / +9.7% | 1.00 | 1 | - |
| SPY price only | - | +0.1% (-22.9 to +27.1) | -20.5 to +22.1 | -0.07 | -25.4% | -20.3% / +7.9% | 1.00 | 1 | - |

Rotation control of sim1 (100 rotations): median Sharpe -0.11 (95% -0.79 to 0.80); share at or above the actual 0.01. Vector view of sim1: 1469 bets of 100, hit rate 55.5%, mean net +0.52 per bet.

Strategy carried to the test period: **sim3 on LECTURE+TIME+SPACE** (validation Sharpe 1.79).

Pullback buying (sim5), money-weighted annual return, one unit deposited each month:

| Rule | BTC-USD | ETH-USD | GLD | QQQ | SPY | Mean |
|---|---:|---:|---:|---:|---:|---:|
| lump_sum | -5.9% | -22.3% | +6.6% | +1.7% | +1.3% | -3.7% |
| monthly | +51.7% | +25.6% | +9.7% | +24.9% | +14.8% | +25.3% |
| cash | +4.5% | +4.5% | +4.5% | +4.5% | +4.5% | +4.5% |
| atr_1 | +51.6% | +25.2% | +9.9% | +25.2% | +15.0% | +25.4% |
| atr_2 | +52.2% | +25.8% | +10.1% | +24.9% | +15.2% | +25.6% |
| atr_3 | +53.1% | +26.1% | +10.3% | +24.1% | +15.1% | +25.7% |
| prior_month_low | +59.3% | +30.5% | +9.5% | +27.0% | +15.9% | +28.4% |

Trigger carried to the test period: **prior_month_low**.

## 5. The test period, read once
`python -m tsip.capstone.evaluate` refuses to run from uncommitted code or a second time; its run is logged in `registry/capstone_test_runs.json`. The notebook only displays the saved result.

In [8]:
summary_path = results / "test_h5.json"
walk_path = results / "test_walk_forward.json"
if summary_path.exists():
    walk = json.loads(walk_path.read_text()) if walk_path.exists() else None
    display(Markdown(evaluate.markdown(json.loads(summary_path.read_text()), walk)))
else:
    print("Not run yet: python -m tsip.capstone.evaluate, then --strategies")
print(json.dumps(json.loads(Path(evaluate.RUN_LOG).read_text()), indent=2))

Rows: training 19,534, validation 2,938, test 4,212 (test base rate 55.1% positive).

| Feature set | Test AUC | Validation AUC (refit, check) |
|---|---:|---:|
| LECTURE | 0.552 | 0.555 |
| LECTURE+TIME | 0.534 | 0.561 |
| LECTURE+SPACE | 0.547 | 0.548 |
| LECTURE+TIME+SPACE | 0.540 | 0.553 |

| Statistic | Lift over LECTURE | 95% interval |
|---|---:|---|
| **Registered: LECTURE+TIME+SPACE** | **-0.0122** | -0.0394 to +0.0160 |
| LECTURE+TIME (exploratory) | -0.0182 | -0.0413 to +0.0059 |
| LECTURE+SPACE (exploratory) | -0.0045 | -0.0195 to +0.0112 |
| Control: TIME and SPACE shuffled | -0.0209 | -0.0416 to -0.0015 |
| Control: random walk | -0.0034 | -0.0232 to +0.0165 |

H-2026-016: **rejected** (lift at least 0.02: no; interval excludes zero: no; permuted control shows no lift: yes; random walk control shows no lift: yes).

**Strategies on LECTURE** (test period 2024-01-01 to 2026-11-03, frozen parameters)

| Strategy | CAGR (95%) | CAGR minus five equal (95%) | CAGR minus SPY with dividends (95%) | Sharpe | Max drawdown | Exposure | Entries | One-session delay: CAGR, Sharpe |
|---|---|---|---|---:|---:|---:|---:|---|
| sim1 | +8.3% (+2.3% to +15.3%) | -47.6% to +21.0% | -28.6% to +4.7% | 0.50 | -9.4% | 0.06 | 77 | +10.7%, 0.76 |
| sim2 | +8.3% (+2.3% to +15.3%) | -47.6% to +21.0% | -28.6% to +4.7% | 0.50 | -9.4% | 0.06 | 77 | +10.7%, 0.76 |
| sim3 | +5.9% (+1.9% to +10.2%) | -50.3% to +18.6% | -31.2% to +2.4% | 0.42 | -5.6% | 0.03 | 60 | +6.3%, 0.57 |
| sim4 | -8.0% (-28.1% to +13.8%) | -71.8% to +15.2% | -53.4% to +0.6% | -0.58 | -36.3% | 0.11 | 92 | +4.7%, 0.12 |
| buy_hold_equal | +17.7% (-12.6% to +56.0%) | - | -26.2% to +30.0% | 0.59 | -26.4% | 1.00 | 5 | - |
| buy_hold_crypto | +11.4% (-40.3% to +100.9%) | -32.2% to +50.0% | -57.1% to +75.4% | 0.39 | -58.6% | 1.00 | 2 | - |
| spy_total_return | +19.0% (+2.0% to +38.4%) | -30.0% to +26.2% | - | 0.93 | -18.8% | 1.00 | 1 | - |
| spy_price_only | +17.7% (+0.9% to +36.7%) | -31.3% to +24.5% | -2.1% to -0.7% | 0.86 | -19.0% | 1.00 | 1 | - |

Rotation control of sim1: median Sharpe 0.31 (95% -0.63 to 1.17); share at or above the actual 0.38.

**Strategies on LECTURE+TIME+SPACE** (test period 2024-01-01 to 2026-11-03, frozen parameters)

| Strategy | CAGR (95%) | CAGR minus five equal (95%) | CAGR minus SPY with dividends (95%) | Sharpe | Max drawdown | Exposure | Entries | One-session delay: CAGR, Sharpe |
|---|---|---|---|---:|---:|---:|---:|---|
| sim1 | +7.8% (-5.5% to +23.0%) | -43.5% to +18.4% | -28.0% to +4.7% | 0.32 | -16.2% | 0.33 | 299 | +8.5%, 0.37 |
| sim2 | +7.8% (-5.5% to +23.0%) | -43.5% to +18.4% | -28.0% to +4.7% | 0.32 | -16.2% | 0.33 | 299 | +8.5%, 0.37 |
| sim3 | +0.7% (-9.9% to +13.2%) | -51.4% to +11.0% | -36.4% to -1.4% | -0.35 | -13.8% | 0.22 | 233 | +1.8%, -0.24 |
| sim4 | -7.5% (-44.7% to +57.0%) | -93.7% to +63.5% | -71.7% to +43.6% | -0.07 | -54.7% | 0.66 | 261 | -22.3%, -0.45 |
| buy_hold_equal | +17.7% (-12.6% to +56.0%) | - | -26.2% to +30.0% | 0.59 | -26.4% | 1.00 | 5 | - |
| buy_hold_crypto | +11.4% (-40.3% to +100.9%) | -32.2% to +50.0% | -57.1% to +75.4% | 0.39 | -58.6% | 1.00 | 2 | - |
| spy_total_return | +19.0% (+2.0% to +38.4%) | -30.0% to +26.2% | - | 0.93 | -18.8% | 1.00 | 1 | - |
| spy_price_only | +17.7% (+0.9% to +36.7%) | -31.3% to +24.5% | -2.1% to -0.7% | 0.86 | -19.0% | 1.00 | 1 | - |

Rotation control of sim1: median Sharpe 0.46 (95% -0.47 to 1.19); share at or above the actual 0.63.

Benchmark comparison (registered strategy sim3 on LECTURE+TIME+SPACE): CAGR +0.7% against +17.7% (five equal) and +19.0% (SPY with dividends): **not above both**; the rubric's exceptional-returns point is not claimed.

Pullback buying, money-weighted annual return (frozen trigger: prior_month_low):

| Rule | BTC-USD | ETH-USD | GLD | QQQ | SPY | Mean |
|---|---:|---:|---:|---:|---:|---:|
| lump_sum | +20.1% | -3.6% | +27.3% | +20.7% | +19.0% | +16.7% |
| monthly | -0.1% | -13.5% | +20.4% | +17.6% | +16.4% | +8.2% |
| cash | +4.0% | +4.0% | +4.0% | +4.0% | +4.0% | +4.0% |
| atr_1 | +0.2% | -13.4% | +20.1% | +17.7% | +16.4% | +8.2% |
| atr_2 | +0.5% | -13.3% | +20.1% | +17.7% | +16.3% | +8.3% |
| atr_3 | +0.3% | -13.5% | +19.6% | +17.3% | +16.1% | +7.9% |
| prior_month_low (frozen) | -3.9% | -16.7% | +18.7% | +16.1% | +15.5% | +5.9% |

Walk-forward check (exploratory; refitted at the start of each test year on all earlier rows):

| Horizon | AUC LECTURE | AUC +TIME+SPACE | Lift (95%) |
|---|---:|---:|---|
| 5 rows | 0.533 | 0.526 | -0.0075 (-0.0329 to +0.0178) |
| 20 rows | 0.588 | 0.571 | -0.0173 (-0.0470 to +0.0141) |

[
  {
    "kind": "registered",
    "started_utc": "2026-11-10T12:06:34+00:00",
    "commit": "393631ee4a14cbf0b613b2f9d57b1c2bfe1b46f1",
    "data_sha256": {
      "prices": "6e3132b4402f8337a51adb8014c79b79476c7ef08270ef989291998381e1b3fb",
      "fred": "924cb85fa901250317ed8cbfeb5b5084b122bb8a21cd4bf6d897c31e24916669"
    },
    "rerun_reason": null,
    "status": "completed",
    "outputs": [
      "test_models_h5.json",
      "test_h5.json"
    ]
  },
  {
    "kind": "walk_forward_h5",
    "started_utc": "2026-11-10T12:09:21+00:00",
    "commit": "2c537893997251c14e7a15a3dbbf9e37c90b4d20",
    "data_sha256": {
      "prices": "6e3132b4402f8337a51adb8014c79b79476c7ef08270ef989291998381e1b3fb",
      "fred": "924cb85fa901250317ed8cbfeb5b5084b122bb8a21cd4bf6d897c31e24916669"
    },
    "rerun_reason": null,
    "status": "completed",
    "outputs": [
      "test_walk_forward.json"
    ]
  },
  {
    "kind": "walk_forward_h20",
    "started_utc": "2026-11-10T12:10:30+00:00",
    "com

## Diagnostics: where the model works and where it fails
The vector view's return on capital, test-period AUC and precision by instrument and by month of the year, calibration and the most confident wrong predictions (exploratory; the reading and the ideas for improvement are in `docs/CAPSTONE.md`).

In [9]:
diagnostics = analysis.markdown(store)
display(Markdown("**Vector view" + diagnostics.split("**Vector view", 1)[1]))

**Vector view, approximate return on capital** (bets of 100; total net over the capital needed)

| Period | Feature set | Bets | Hit rate | Total net | Capital needed | Return on capital |
|---|---|---:|---:|---:|---:|---:|
| validation | LECTURE | 294 | 55.1% | +241 | 3,200 | +7.5% |
| validation | LECTURE+TIME+SPACE | 1,469 | 55.5% | +769 | 3,400 | +22.6% |
| test | LECTURE | 238 | 63.0% | +262 | 3,400 | +7.7% |
| test | LECTURE+TIME+SPACE | 1,328 | 56.7% | +463 | 3,400 | +13.6% |

**Test period by instrument** (direction of the next 5 rows; precision of the 10% most confident calls, cut fixed on the whole test set; exploratory)

| Instrument | Rows | Base rate | AUC LECTURE | AUC +TIME+SPACE | Precision top decile LECTURE (rows) | Precision top decile +TIME+SPACE (rows) |
|---|---:|---:|---:|---:|---|---|
| BTC-USD | 1038 | 51.3% | 0.509 | 0.515 | 57.9% (38) | 58.2% (55) |
| ETH-USD | 1038 | 49.3% | 0.514 | 0.526 | 53.8% (52) | 48.5% (101) |
| GLD | 712 | 60.1% | 0.556 | 0.568 | 76.1% (46) | 68.9% (45) |
| QQQ | 712 | 58.6% | 0.536 | 0.515 | 63.4% (131) | 59.0% (100) |
| SPY | 712 | 60.8% | 0.554 | 0.523 | 65.8% (155) | 66.1% (121) |

**Test period by month of the year** (direction of the next 5 rows; precision of the 10% most confident calls, cut fixed on the whole test set; exploratory)

| Month of the year | Rows | Base rate | AUC LECTURE | AUC +TIME+SPACE | Precision top decile LECTURE (rows) | Precision top decile +TIME+SPACE (rows) |
|---|---:|---:|---:|---:|---|---|
| 1 | 369 | 53.1% | 0.660 | 0.546 | 85.0% (20) | 62.8% (43) |
| 2 | 344 | 58.1% | 0.643 | 0.661 | 68.2% (22) | 65.5% (29) |
| 3 | 375 | 52.0% | 0.466 | 0.427 | 44.9% (118) | 43.7% (71) |
| 4 | 372 | 62.9% | 0.592 | 0.642 | 80.7% (83) | 76.3% (93) |
| 5 | 375 | 55.5% | 0.484 | 0.510 | 74.3% (35) | 78.6% (14) |
| 6 | 360 | 49.7% | 0.727 | 0.767 | 100.0% (19) | 100.0% (9) |
| 7 | 384 | 54.2% | 0.443 | 0.469 | 46.2% (26) | 59.6% (47) |
| 8 | 378 | 55.3% | 0.553 | 0.528 | 70.0% (20) | 47.4% (19) |
| 9 | 366 | 65.6% | 0.550 | 0.538 | 78.6% (42) | 88.2% (17) |
| 10 | 390 | 42.3% | 0.479 | 0.431 | 35.7% (28) | 34.4% (64) |
| 11 | 246 | 58.5% | 0.537 | 0.542 | 37.5% (8) | 60.0% (15) |
| 12 | 253 | 56.9% | 0.525 | 0.642 | 100.0% (1) | 0.0% (1) |

**Wrong predictions** (LECTURE+TIME+SPACE): error rate 52.5%. Share of the 10% most confident errors by instrument: BTC-USD 43%, ETH-USD 38%, GLD 11%, QQQ 4%, SPY 4%; by year: 2024 30%, 2025 34%, 2026 35%.

Calibration (mean predicted probability against the base rate): BTC-USD 37.7% against 51.3%; ETH-USD 39.2% against 49.3%; GLD 40.2% against 60.1%; QQQ 44.6% against 58.6%; SPY 45.4% against 60.8%.

| Date | Instrument | Probability | Outcome (positive 5 rows) | Return of the next 5 rows |
|---|---|---:|---:|---:|
| 2025-05-08 | ETH-USD | 0.145 | 1 | +21.5% |
| 2026-08-22 | ETH-USD | 0.156 | 1 | +3.6% |
| 2026-03-01 | ETH-USD | 0.173 | 1 | +2.0% |
| 2026-03-07 | ETH-USD | 0.175 | 1 | +5.3% |
| 2026-03-07 | BTC-USD | 0.176 | 1 | +4.8% |
| 2025-05-24 | ETH-USD | 0.183 | 1 | +4.0% |
| 2025-05-11 | ETH-USD | 0.185 | 1 | +1.0% |
| 2026-03-06 | BTC-USD | 0.187 | 1 | +3.0% |
| 2026-03-06 | ETH-USD | 0.188 | 1 | +3.7% |
| 2026-03-08 | BTC-USD | 0.188 | 1 | +7.6% |

## 6. Broker example (paper trading, educational)
The orders strategy sim1 would place from the latest prediction, printed (dry run). `python -m tsip.capstone.paper_order --send` submits them to Alpaca's paper endpoint with simulated money. It is not a signal: the strategy did not beat buy and hold out of sample.

In [10]:
paper_order.main(["--notional", "1000"])

{
  "as_of": "2026-11-09",
  "threshold": 0.44845043437653953,
  "probabilities": {
    "ETH-USD": 0.5234694499716778,
    "BTC-USD": 0.5248096705610847,
    "GLD": 0.48358500854155256,
    "QQQ": 0.5272265262538505,
    "SPY": 0.5702302141996222
  },
  "orders": [
    {
      "symbol": "SPY",
      "notional": "200.00",
      "side": "buy",
      "type": "market",
      "time_in_force": "day",
      "client_order_id": "tsip-course-example-2026-11-09-SPY"
    },
    {
      "symbol": "QQQ",
      "notional": "200.00",
      "side": "buy",
      "type": "market",
      "time_in_force": "day",
      "client_order_id": "tsip-course-example-2026-11-09-QQQ"
    },
    {
      "symbol": "GLD",
      "notional": "200.00",
      "side": "buy",
      "type": "market",
      "time_in_force": "day",
      "client_order_id": "tsip-course-example-2026-11-09-GLD"
    }
  ]
}
Educational paper-trading example; the strategy did not beat buy and hold out of sample.


## Reading

The written conclusions, the controls and the limits are in `docs/CAPSTONE.md` (steps 3 to 5). In short: the hypothesis was registered before training, every choice was made on validation, the test period was read once, and the time–space features did not improve the prediction there; the strategies that looked strong on validation did not beat buy and hold. The next study, fixed before any figure, asks a narrower question: when to buy with a fixed monthly budget (`specs/0029-when-to-buy-with-a-monthly-budget.md`).